# Signal Forge Strategy Review — Reproducible Audit

只读冻结快照；不联网重取，不训练或修改策略。运行目录为本笔记本所在文件夹。

In [1]:
from pathlib import Path
import json
from collections import Counter
from review_analysis import compute
DATA = Path('review-extract.json')
data = json.loads(DATA.read_text(encoding='utf-8'))
results = compute(data)
assert results == json.loads(Path('analysis-results.json').read_text(encoding='utf-8'))
for strategy, result in results.items():
    print(strategy, result['overall'])

fishing-reversal {'events': 30, 'days': 11, 'securities': 30, 'known': 28, 'unknown': 2, 'up': 12, 'up_pct': 42.86, 'mean_pct': 0.0462, 'median_pct': -0.4911, 'min_pct': -6.2859, 'max_pct': 7.7177}
limit-up-ambush {'events': 351, 'days': 12, 'securities': 279, 'known': 351, 'unknown': 0, 'up': 143, 'up_pct': 40.74, 'mean_pct': -0.6067, 'median_pct': -0.3667, 'min_pct': -11.2335, 'max_pct': 8.9895}


## 预先指定的描述分组

不是最优阈值搜索，也不是未来验证。使用首次评分，不使用事后最高评分或命中次数。

In [2]:
ambush = results['limit-up-ambush']
for key in ['score_lt75','score_75_85','score_ge85','early_period','late_period','late_score_lt75','late_score_ge85','morning','afternoon']:
    print(key, ambush['cuts'][key])
print('exploratory correlations', ambush['correlations_exploratory_not_causal'])

score_lt75 {'events': 158, 'days': 12, 'securities': 151, 'known': 158, 'unknown': 0, 'up': 67, 'up_pct': 42.41, 'mean_pct': -0.2022, 'median_pct': -0.1126, 'min_pct': -6.9802, 'max_pct': 8.9895}
score_75_85 {'events': 150, 'days': 11, 'securities': 130, 'known': 150, 'unknown': 0, 'up': 58, 'up_pct': 38.67, 'mean_pct': -0.8076, 'median_pct': -0.5651, 'min_pct': -11.2335, 'max_pct': 8.8153}
score_ge85 {'events': 43, 'days': 10, 'securities': 40, 'known': 43, 'unknown': 0, 'up': 18, 'up_pct': 41.86, 'mean_pct': -1.3917, 'median_pct': -0.4124, 'min_pct': -9.0525, 'max_pct': 4.7157}
early_period {'events': 189, 'days': 7, 'securities': 167, 'known': 189, 'unknown': 0, 'up': 89, 'up_pct': 47.09, 'mean_pct': -0.1758, 'median_pct': -0.0543, 'min_pct': -7.0042, 'max_pct': 8.9895}
late_period {'events': 162, 'days': 5, 'securities': 142, 'known': 162, 'unknown': 0, 'up': 54, 'up_pct': 33.33, 'mean_pct': -1.1093, 'median_pct': -0.7651, 'min_pct': -11.2335, 'max_pct': 4.9969}
late_score_lt75 {'e

In [3]:
for strategy, result in results.items():
    print(strategy, 'versions', result['versions'], 'flags', result['quality_flags'])
    print('patterns', result['patterns'], 'unknown', result['unknown_next_day_status'])
    rows = [r for r in data['records'] if r['strategy']==strategy]
    assert len(rows)==len(set((r['strategy'],r['date'],r['code']) for r in rows))
    print('limit labels', Counter(str(r['post_alert_limit']) for r in rows))
print('All source denominator and snapshot reproducibility checks passed.')

fishing-reversal versions {'1.1.0': 30} flags {'publication_time_unknown': 30, 'configuration_snapshot_unknown': 30, 'no_post_alert_observation': 30, 'limit_price_unknown': 30}
patterns {'no_recovery_above_reference': 5, 'closed_below_signal_low': 4, 'rebound_not_held': 11} unknown {'not_yet_closed': 2}
limit labels Counter({'None': 30})
limit-up-ambush versions {'0.2.0': 351} flags {'publication_time_unknown': 351, 'configuration_snapshot_unknown': 351, 'limit_price_unknown': 351}
patterns {} unknown {}
limit labels Counter({'None': 351})
All source denominator and snapshot reproducibility checks passed.


In [4]:
from review_sql_views import query_views
alerts, bands = query_views(data)
artifact=json.loads(Path('report-artifact.json').read_text(encoding='utf-8'))
assert alerts == artifact['snapshot']['datasets']['alerts']
assert bands == artifact['snapshot']['datasets']['bands']
print('Chart and table SQL outputs exactly match the report snapshot.',len(alerts),len(bands))

Chart and table SQL outputs exactly match the report snapshot. 351 3


## 边界

已入选样本、少量相关日期、缺失精确涨停价和逐次配置。V1 次日结果与涨停伏击当日结果不能混为同一个交易胜率。未来测试必须按时间保留，不可随机拆同日股票以宣称独立验证。